# 二、经营指标体系设计与SQL分析
建立一套有业务意义的电商经营指标体系。
## 2.1 建立经营指标体系
将项目的经营指标体系划分为 六大分析模块、约 30 个核心指标
| 模块 | 核心 KPI | 辅助分析 |
|---|---|---|
| 整体经营 | 销售额、有效订单量、客单价、取消率 | 月度增长、商品件数、支付金额 |
| 商品经营 | 商品销售额、毛利额、毛利率 | 品类、品牌、折扣、商品四象限 |
| 客户价值 | 客户数、复购率、客户平均消费金额 | RFM、客户分群、客户贡献 |
| 客户留存 | 次月留存率、各期留存率 | Cohort 热力图 |
| 支付与订单 | 支付方式占比、平均配送时长、平均评分 | 分期、支付金额、配送延迟 |
| 物流与卖家 | 卖家销售额、卖家订单量、区域分布 | 跨州订单、卖家表现 |
### 2.1.1 整体经营指标体系
判断平台整体经营规模、销售变化和订单表现。
| 指标 | 计算公式 | 数据表 | 说明 |
|---|---|---|---|
| 商品销售额 | Σ商品成交单价 × 商品数量（每行商品明细为一件时直接求和） | order_items | 衡量平台商品交易规模 |
| 订单总量 | COUNT(DISTINCT order_id) | order | 衡量订单规模 |
| 有效订单量 | COUNT(DISTINCT order_id)，筛选已发货订单 | order | 衡量有效履约订单规模 |
| 取消订单量 | COUNT(DISTINCT order_id)，筛选已取消订单 | order | 衡量订单取消规模 |
| 订单取消率 | 取消订单量 / 订单总量 × 100% | order | 衡量订单取消情况 |
| 客户总数 | COUNT(DISTINCT customer_unique_id) | customers | 衡量实际客户规模 |
| 客单价（AOV） | 商品销售额 / 有效订单量 | order、order_items | 衡量每笔有效订单平均商品消费金额 |
| 平均订单商品件数 | 商品总件数 / 有效订单量 | order_items、order | 衡量订单商品购买数量 |
| 月度销售增长率 | （本月销售额－上月销售额）/ 上月销售额 × 100% | order、order_items | 观察销售增长情况 |
| 平均订单支付金额 | 支付总金额 / 支付订单数 | order_payments | 衡量客户实际支付规模 |


In [ ]:
# 2.2.1整体经营指标体系
# 1. 商品销售额
select sum(order_items.price) as total_price
from order_items;
# 2. 订单总量
select count(distinct orders.order_id) as total_orders
from orders;
# 3. 有效订单量
select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered');
# 4. 取消订单量
select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('canceled');
# 5. 订单取消率
select round(count(distinct case when orders.order_status = 'canceled' then order_id end)*100/
count(distinct orders.order_id),2) as vaild_per
from orders;
# 6. 客户总数
select count(distinct customers.customer_unique_id) as total_cus
from customers;
# 7. 客单价
with sales as(select sum(order_items.price) as total_price
from order_items),

vaildorders as (select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered'))
select round(s.total_price/v.vaild_orders,2) as aov
from sales s
cross join vaildorders v;

# 8. 平均订单商品总数
with num_items as (select count(*) as total_items from order_items),
vaildorders as (select count(distinct orders.order_id) as vaild_orders
from orders
where order_status in ('shipped', 'delivered'))
select round(n.total_items/v.vaild_orders,0) as ave_item
from num_items n
cross join vaildorders v;

# 9. 月度销售增长率

with monthlysales as(select date_format(o.order_purchase_timestamp, '%Y-%m') as sales_month,
sum(oi.price) as monthly_price
from orders o 
join order_items oi on o.order_id = oi.order_id
where o.order_status not in ('canceled', 'unavailable')
group by date_format(o.order_purchase_timestamp, '%Y-%m')),
saleswithlag as (select sales_month,monthly_price,lag(monthly_price) over (order by sales_month) as prev_month_sales
from monthlysales)
select round((monthly_price - prev_month_sales) / prev_month_sales * 100, 2) as growth_rate_percentage
FROM saleswithlag;

# 10.平均订单支付额

SELECT 
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS avg_order_payment
FROM order_payments;



### 2.1.2 商品经营分析指标
| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 实际客户数 | COUNT(DISTINCT customer_unique_id) | customers、order | 统计实际客户数量 |
| 新客户数 | 首次购买时间落在统计周期内的客户数 | customers、order | 衡量新客户获取 |
| 复购客户数 | 购买订单数 ≥ 2 的客户数 | customers、order | 衡量客户重复购买情况 |
| 客户复购率 | 复购客户数 / 有购买行为客户数 × 100% | customers、order | 衡量客户复购情况 |
| 客户平均消费金额 | 商品销售额 / 实际购买客户数 | customers、order、order_items | 衡量客户平均消费水平 |
| 客户平均购买频次 | 有效订单量 / 实际购买客户数 | customers、order | 衡量客户购买频率 |
| 客户生命周期价值（简化） | 客户历史累计商品消费金额 | customers、order、order_items | 衡量客户累计贡献 |
| R（Recency） | 统计截止日期－客户最近一次购买日期 | customers、order | 衡量客户最近购买活跃程度 |
| F（Frequency） | 客户在统计周期内的有效订单数量 | customers、order | 衡量客户购买频次 |
| M（Monetary） | 客户在统计周期内的累计商品消费金额 | customers、order、order_items | 衡量客户消费贡献 |


**RFM 维度设计**

| 维度 | 含义 | 评分方向 |
|---|---|---|
| R | 最近一次购买距今的时间 | 越短分数越高 |
| F | 购买频次 | 越高分数越高 |
| M | 累计消费金额 | 越高分数越高 |

**RFM 客户分群**

| 客户群体 | 典型特征 | 运营建议 |
|---|---|---|
| 重要价值客户 | R、F、M 均较高 | 重点维护、会员权益 |
| 重要发展客户 | R 较高，F、M 有提升空间 | 促进复购和消费升级 |
| 重要保持客户 | F、M 较高，但 R 较低 | 定向召回 |
| 一般价值客户 | 消费频次和金额较低 | 低成本促活 |
| 流失风险客户 | 长时间未购买 | 召回优惠、定向营销 |


In [ ]:
# 2.1.2 商品经营分析指标
# 定义临时变量
SET @start_date = '2024-01-01';
SET @end_date   = '2024-12-31';
SET @as_of_date = '2025-01-01';
## 有效订单
with vaild_orders as (select o.order_id,c.customer_unique_id,date(o.order_purchase_timestamp) as purchase_date
from orders o
join customers c on c.customer_id = o.customer_id
where o.order_status not in ('canceled', 'unavailable')),
order_amt as (select order_items.order_id, sum(price) as item_amt
from order_items
group by order_id),
vaild_order_amt as (select v.order_id,v.customer_unique_id,v.purchase_date, coalesce(a.item_amt) as item_amount
from vaild_orders v
left join order_amt a on a.order_id = v.order_id),
period_order as (select * from vaild_order_amt
where purchase_date >= @start_date and purchase_date <= @end_date),
period_customer as (select customer_unique_id,count(distinct order_id) as freq,sum(item_amount) as monetary
from period_order group by customer_unique_id),
first_purchase as(select customer_unique_id,min(purchase_date) as first_purchase_date
from vaild_order_amt group by customer_unique_id),
new_customer as (select customer_unique_id from first_purchase
where first_purchase_date >= @start_date
and first_purchase_date <= @end_date),
repurchase_customer as (select customer_unique_id from period_customer 
where freq >=2),
clv as (select customer_unique_id, sum(item_amount) as clv_amount, max(purchase_date) as lastest_pur_date
from vaild_order_amt
order by customer_unique_id)
# 1. 有效客户数
select count(*) as real_customers from period_customer;
# 2. 新客户数
select count(*) as new_customers from new_customer;
# 3. 复购客户数
select count(*) as repurchase_customers from repurchase_customer;
# 4. 客户复购率
select round((select count(*) as repurchase_customers from repurchase_customer)*100/
(select count(*) as real_customers from period_customer),2) as rep_per;
# 5. 客户平均消费金额
select round((select sum(monetary) from period_customer)/
(select count(*) as real_customers from period_customer),2) as ave_price;
# 6. 客户平均购买频次
select round((select sum(freq) from period_customer)/ 
nullif((select count(*) from period_customer), 0), 2);
# 7. 客户生命周期价值
SELECT p.customer_unique_id,DATEDIFF(@as_of_date, h.last_purchase_date) AS R_day,p.freqAS F,p.monetaryAS M,
COALESCE(h.clv_amount, 0)AS CLV
FROM period_customer p
JOIN history_customer h
  ON h.customer_unique_id = p.customer_unique_id
ORDER BY R_day ASC, F DESC, M DESC;

### 2.1.3 客户留存分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 首购客户数 | 某月首次购买客户数量 | customers、order | 衡量新客户获取 |
| 次月留存客户数 | 某月首购客户中，下月再次购买的客户数 | customers、order | 衡量次月留存 |
| 次月留存率 | 次月留存客户数 / 首购客户数 × 100% | customers、order | 衡量客户次月留存表现 |
| 第N月留存率 | 首购月份为第0期的客户中，第N期仍有购买行为的客户数 / 首购客户数 | customers、order | 衡量长期留存 |
| 客户流失率（观察口径） | 观察期内未复购客户数 / 符合观察条件的客户数 | customers、order | 衡量客户流失情况 |


Cohort 分析设计
按照客户第一次购买的月份划分 Cohort。


### 2.1.4 支付与订单分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 各支付方式订单数 | 按 payment_type 统计 DISTINCT order_id | order_payments | 了解支付方式偏好 |
| 各支付方式使用占比 | 使用某支付方式的订单数 / 有支付记录的订单数 | order_payments | 比较支付方式结构 |
| 各支付方式支付金额 | 按 payment_type 汇总 payment_value | order_payments | 比较不同支付方式交易规模 |
| 平均分期期数 | AVG(payment_installments) | order_payments | 分析消费者分期习惯 |
| 多次支付订单数 | COUNT(DISTINCT order_id)，筛选 payment_sequential 大于1的订单 | order_payments | 识别多笔支付订单 |
| 支付审批时长 | order_approved_at－order_purchase_timestamp | order | 分析付款审批效率 |
| 发货等待时间 | order_delivered_carrier_date－order_approved_at | order | 衡量订单发货效率 |
| 物流配送时长 | order_delivered_customer_date－order_delivered_carrier_date | order | 衡量物流配送效率 |
| 配送延迟天数 | 实际送达时间－预计送达时间 | order | 衡量履约表现 |
| 平均客户评分 | AVG(review_score) | order_reviews | 衡量客户满意度 |
### 2.1.5 物流与卖家分析指标

| 指标名称 | 计算公式 | 数据来源 | 分析目的 |
|---|---|---|---|
| 卖家总数 | COUNT(DISTINCT seller_id) | sellers | 衡量卖家规模 |
| 卖家订单量 | COUNT(DISTINCT order_id) | sellers、order_items | 衡量卖家订单贡献 |
| 卖家商品销售额 | SUM(order_items.price) | sellers、order_items | 衡量卖家销售规模 |
| 卖家商品毛利贡献 | Σ（商品成交价－商品成本） | sellers、order_items、products | 比较卖家关联商品的毛利贡献 |
| 卖家平均订单金额 | 卖家商品销售额 / 卖家有效订单数 | sellers、order_items、order | 比较卖家订单价值 |
| 卖家区域分布 | 按 seller_state 统计卖家数 | sellers | 分析卖家地域结构 |
| 客户区域分布 | 按 customer_state 统计客户数 | customers | 分析客户地域结构 |
| 跨州订单占比 | 客户州与卖家州不同的订单数 / 有效订单数 | customers、sellers、order_items、order | 分析跨区域交易情况 |
| 平均配送延迟 | 实际送达时间－预计送达时间 | order | 衡量平台整体配送表现 |